# Sawtooth oscillation - verification

Use the **Python (FAITH labelmaker)** kernel, started under the wrapper:
`pixi run -e labelmaker fdp run jupyter lab`. Set `shot` below, drag a time
range on any panel, then press *Mark present* / *Mark absent*, *Verify* and
*Save*.

**Eight of the ten rostered shots are not in the corpus** (178640, 178641,
178642, 179310, 180090, 180406, 180627, 184084 all predate it) and are
fetched live over MDSplus instead. The first fetch of a shot takes several
minutes PER CHANNEL - the panel cell below reads five channels, so budget
something like ten minutes the first time you open an out-of-corpus shot.
After that it is cached under
`data/events/sawtooth_oscillation/review/_cache/<shot>_ece.npz` and reopening
the same shot is instant. That cache is a fetch cache, not a review record:
it holds the raw record verbatim, costs about **62 MB per fetched shot**, has
no cap and no eviction, and is safe to delete at any time - the next open of
that shot simply refetches. `*.npz` is gitignored, so it cannot be committed
by accident. Only 192238 and 195032 are in the corpus, which is why the
default `shot` below is 192238: this notebook opens with no fetch at all on
first run.

**What you are looking for**: a sawtooth crash is a FAST drop on the core
ECE channels - order 100 microseconds to 1 ms, which is why this notebook
reads the raw 500 kHz record rather than the feature store's 1 ms-decimated
`ece` (physically too coarse to show a crash shaped like this). The
**inversion radius** is the point where the crash flips sign between rows -
channels inside q=1 drop, channels outside it rise - and that sign flip is
what tells a sawtooth apart from any other fast transient. The four panels
below are grouped in blocks of four adjacent channels so that flip is
visible row to row.

**No saved label grid exists for this category.** No detector has ever run
for sawtooth, so `format/shots/` holds nothing - the directory does not even
exist - and `review()` below warns `no saved label grid`, writes
`NO LABEL ROW` into the figure title and leaves the label row out. That is
expected, not a bug, and it holds for all ten rostered shots: your own marks
are this category's first labels. If an `ece_sawtooth` detector run lands
later it will write under `format/shots/`, which is the `source` already set
below, and the label row will appear on its own.

`tier` and `holdout` in `shots.csv` are curation calls set by hand; *Save*
records the reviewer and the date and does not promote anything.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from labeler.config import Paths
from labeler.events.verify import (
    ECE_POINT,
    NoDataError,
    Panel,
    corpus_signal,
    fdp_signal,
    review,
)

event = "sawtooth_oscillation"
shot = 192238  # in the corpus; replace with any shot from shots.csv
source = "format/shots"

# 600 ms at 500 kHz is 300,000 points per channel, which still plots
# comfortably. Once a shot is cached, WIDENING this window is what costs -
# the fetch itself is already paid for and cached under review/_cache/.
t_range = (2000.0, 2600.0)

# Four rows of four ADJACENT channels covering 20-35, sixteen in all - free
# from the corpus,
# so read in full there. A live fdp fetch is not free (minutes per
# channel), so an out-of-corpus shot instead reads this five-channel SAMPLE
# of the same span; a reviewer on a fetched shot is looking at 5 channels,
# not 16, and that difference is deliberate, not a bug.
corpus_rows = [(20, 21, 22, 23), (24, 25, 26, 27), (28, 29, 30, 31), (32, 33, 34, 35)]
sampled_channels = (20, 24, 28, 32, 36)


def corpus_panel(row):
    """One row of four adjacent ECE channels, overplotted to show the flip."""
    array = corpus_signal(shot, "ece", channels=row, t_range=t_range)
    return Panel(
        title=f"ECE ch {row[0]}-{row[-1]}",
        x=array.x,
        y=array.y,
        ylabel="keV",
        legend=[f"ch {c}" for c in row],
    )


try:
    panels = [corpus_panel(row) for row in corpus_rows]
except NoDataError:
    # 178640-184084 predate the corpus. ~80-145 s per channel on the first
    # fetch, then cached under review/_cache/ - ~62 MB a shot, uncapped, and
    # safe to delete whenever the disk matters more than the wait.
    cache = Paths.from_env().label_tables / event / "review" / "_cache"
    ece = fdp_signal(
        shot,
        [ECE_POINT.format(channel=c) for c in sampled_channels],
        t_range=t_range,
        cache=cache / f"{shot}_ece.npz",
    )
    panels = [
        Panel(
            title="ECE (fetched, 5-channel sample of 20-36)",
            x=ece.x,
            y=ece.y,
            ylabel="keV",
            legend=[f"ch {c}" for c in sampled_channels],
        )
    ]

In [ ]:
session = review(event, shot, panels, source=source)
session

After pressing *Save*, check what was written. Every save writes its own file under `review/`, named
`<shot>__<reviewer>__<stamp>.csv`. Nothing there is ever overwritten or
deleted by this tooling, so a second reviewer cannot destroy the first's
work; the rows are merged into `format/` by hand.

```python
from labeler.events.verify import corrections_for, read_latest_corrections

for path in corrections_for(event, shot):
    print(path.name)
read_latest_corrections(event, shot)
```
